In [138]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
pa_gs = pd.read_csv('data/pa_data.csv').dropna()

coastal_pa_stations = pa_gs[pa_gs['dist_atlantic_km']< 150]
lakeside_pa_stations = pa_gs[pa_gs['dist_greatlakes_km']< 150]
inland_pa_stations = pa_gs[pa_gs['dist_coast_km']>150]

In [17]:
y = pa_gs['growing_season_length']
X = pa_gs[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]

In [226]:
def multiple_regression(input_variables, predicted_variable):
    #performs a standardized multiple linear regression to create a model of y based on the variables within X.
    # Required Libraries: r2score, mean_absolute_error, and mean_squared_error from sklearn.metrics, 
    # train_test_split from sklearn.model_selection, linear_model from sklearn, and StandardScaler from sklearn.preprocessing
    
    #drops NaN values from selected variables
    #input_variables_no_na = input_variables.dropna()
    tot_data_points = int(len(input_variables))
    ## I would prefer that the data cleaning happens within the function both so it's not a prerequisite to run the function
    ## and so the maximum amount of data can be considered for a given regression, but right now the code above makes the
    ## function error out.
    
    #splits data into training and testing groups
    X_train, X_test, y_train, y_test = train_test_split(input_variables, predicted_variable, test_size = 0.2, random_state =0)

    #scales input data to standardize to mean of 0 and standard deviation of 1
    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train)
    X_test_scaled = sc.fit_transform(X_test)
    
    #creates multiple regression model based on training data
    regr = linear_model.LinearRegression()
    regr.fit(X_train_scaled, y_train)

    #calculates mean absolute error, mean squared error, and r squared score based on the predicted y
    y_predicted = regr.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_predicted)
    mse = mean_squared_error(y_test, y_predicted)
    rmse = np.sqrt(mse)
    r_2_score = r2_score(y_test, y_predicted)
    n = len(predicted_variable)
    p = input_variables.shape[1]
    adj_r2_score = 1 - ((1 - r_2_score) * (n - 1) / (n - p - 1))
    y_std = predicted_variable.std()
    coefficients = pd.DataFrame(zip(input_variables.columns, input_variables.std(), regr.coef_))

    coefficients.columns = ['variable', 'var standard dev', 'coefficient']
    
    results = [f'MAE = {mae}', f'MSE = {mse}', f'RMSE = {rmse}', f'R Squared = {r_2_score}', 
               f'Adj. R Squared = {adj_r2_score}', f'Total Data Points: {tot_data_points}', 
               f'(Reference) predicted variable standard deviation = {y_std}', coefficients]
    
    return display(results)

In [227]:
multiple_regression(X, y)

['MAE = 8.653954401679119',
 'MSE = 118.54336216146942',
 'RMSE = 10.887762036409017',
 'R Squared = 0.802483321255483',
 'Adj. R Squared = 0.7956683495914842',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618    -9.448399
 1                   dtr_spring          1.463406    -0.783348
 2                   dtr_summer          1.351448     1.657061
 3                  tmax_annual          1.650279     8.417943
 4               prcp_annual_mm        224.681732   -16.306781
 5       prcp_growing_season_mm        184.123407    20.479302
 6               prcp_spring_mm         85.879985     3.166995
 7                     latitude          0.672409   -11.853968
 8                    longitude          1.749712     7.593680
 9                  elevation_m        158.326516    -1.458976
 10               dist_coast_km         60.

In [228]:
#removed all X variables with coefficient < |1|
X_influential = pa_gs[['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_south_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us','pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'cloud_cover_eastern_us',  'dewpoint_2m_southeast_us','evaporation_southeast_us', 'dewpoint_2m_northeast_us', 
    'soil_moisture_northeast_us','cloud_cover_northeast_us', 'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 
    'dewpoint_station', 'cloud_cover_station']]
multiple_regression(X_influential, y)

['MAE = 8.674296355470146',
 'MSE = 118.47757695625499',
 'RMSE = 10.884740555302868',
 'R Squared = 0.8025929324138605',
 'Adj. R Squared = 0.7978263306641363',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618   -10.279544
 1                   dtr_summer          1.351448     1.818887
 2                  tmax_annual          1.650279     8.367813
 3               prcp_annual_mm        224.681732   -16.398181
 4       prcp_growing_season_mm        184.123407    20.472531
 5               prcp_spring_mm         85.879985     3.244283
 6                     latitude          0.672409   -11.718103
 7                    longitude          1.749712     7.284249
 8                  elevation_m        158.326516    -1.464093
 9                dist_coast_km         60.521467    -1.634493
 10          dist_greatlakes_km         91

In [229]:
# again, removed all X variables with coefficient < |1| from first refined model
X_influential_2 = pa_gs[['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 
    'ohc700_south_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us','pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'dewpoint_2m_southeast_us','evaporation_southeast_us', 'dewpoint_2m_northeast_us', 
    'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 
    'dewpoint_station', 'cloud_cover_station']]
multiple_regression(X_influential_2, y)

['MAE = 8.67312897059619',
 'MSE = 118.4691159883503',
 'RMSE = 10.884351886462984',
 'R Squared = 0.8026070300592203',
 'Adj. R Squared = 0.7983158785387685',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                     variable  var standard dev  coefficient
 0                 dtr_annual          1.162618   -10.279544
 1                 dtr_summer          1.351448     1.818887
 2                tmax_annual          1.650279     8.367813
 3             prcp_annual_mm        224.681732   -16.398181
 4     prcp_growing_season_mm        184.123407    20.472531
 5             prcp_spring_mm         85.879985     3.244283
 6                   latitude          0.672409   -11.718103
 7                  longitude          1.749712     7.284249
 8                elevation_m        158.326516    -1.464093
 9              dist_coast_km         60.521467    -1.634493
 10        dist_greatlakes_km         91.537050   -10.389573
 11  

In [230]:
#removing all x variables with coefficient < |1.5| from 2nd refined model
X_influential_3 = pa_gs[['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 
    'ohc700_south_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us', 'pwat_gulf_coast', 'pwat_station',
    'dewpoint_2m_northeast_us', 'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 'dewpoint_station', 'cloud_cover_station']]
multiple_regression(X_influential_3, y)

['MAE = 8.705701705583804',
 'MSE = 119.92426986972434',
 'RMSE = 10.950994012861313',
 'R Squared = 0.8001824559922244',
 'Adj. R Squared = 0.7963172866180998',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                     variable  var standard dev  coefficient
 0                 dtr_annual          1.162618   -10.098319
 1                 dtr_summer          1.351448     1.625791
 2                tmax_annual          1.650279     9.578128
 3             prcp_annual_mm        224.681732   -16.565525
 4     prcp_growing_season_mm        184.123407    20.563189
 5             prcp_spring_mm         85.879985     3.312616
 6                   latitude          0.672409   -10.149974
 7                  longitude          1.749712     7.232032
 8              dist_coast_km         60.521467    -1.490184
 9         dist_greatlakes_km         91.537050    -9.545822
 10          dist_atlantic_km        133.471354     3.480082
 11

In [231]:
#again, removing all x variables with coefficient < |1.5| from 3nd refined model
X_influential_4 = pa_gs[['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us', 'pwat_gulf_coast', 'pwat_station',
    'dewpoint_2m_northeast_us', 'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 'cloud_cover_station']]
multiple_regression(X_influential_4, y)

['MAE = 8.70360653978436',
 'MSE = 120.27008643481092',
 'RMSE = 10.966771924080984',
 'R Squared = 0.7996062572228855',
 'Adj. R Squared = 0.7963268739395662',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                     variable  var standard dev  coefficient
 0                 dtr_annual          1.162618    -9.830344
 1                 dtr_summer          1.351448     1.397806
 2                tmax_annual          1.650279     9.635027
 3             prcp_annual_mm        224.681732   -16.422984
 4     prcp_growing_season_mm        184.123407    20.348842
 5             prcp_spring_mm         85.879985     3.380801
 6                   latitude          0.672409   -11.649260
 7                  longitude          1.749712    11.966555
 8         dist_greatlakes_km         91.537050   -13.751093
 9           dist_atlantic_km        133.471354     5.966689
 10                oni_annual          0.597267    -2.206270
 11 

In [232]:
X_choice_removals = pa_gs[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'ohc700_atlantic', 'ohc700_atlantic_se', 'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 
    'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us','cloud_cover_eastern_us', 'evaporation_eastern_us', 
    'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us','cloud_cover_northeast_us', 'evaporation_northeast_us', 
    'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania','cloud_cover_pennsylvania', 'evaporation_pennsylvania', 
    'dewpoint_station', 'soil_moisture_station','cloud_cover_station', 'evaporation_station']]
multiple_regression(X_choice_removals, y)

['MAE = 8.659585824165108',
 'MSE = 118.4598136662681',
 'RMSE = 10.883924552580659',
 'R Squared = 0.802622529566985',
 'Adj. R Squared = 0.7974988666176914',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618    -9.448399
 1                   dtr_spring          1.463406    -0.783348
 2                   dtr_summer          1.351448     1.657061
 3                  tmax_annual          1.650279     8.417943
 4               prcp_annual_mm        224.681732   -16.306781
 5       prcp_growing_season_mm        184.123407    20.479302
 6               prcp_spring_mm         85.879985     3.166995
 7                     latitude          0.672409   -11.853968
 8                    longitude          1.749712     7.593680
 9                  elevation_m        158.326516    -1.458976
 10               dist_coast_km         60.5

In [234]:
X_choice_removals_2 = pa_gs[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm','ohc700_atlantic', 'ohc700_north_atlantic','ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world',
    'ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic','sst_gulf_stream', 'sst_tropical_north_atl', 
    'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]
multiple_regression(X_choice_removals_2, y)

['MAE = 8.830461214223291',
 'MSE = 121.79647312838425',
 'RMSE = 11.036143942898908',
 'R Squared = 0.7970629952072209',
 'Adj. R Squared = 0.7927730761393759',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618    -8.580754
 1                   dtr_spring          1.463406    -0.868771
 2                   dtr_summer          1.351448     0.769911
 3                  tmax_annual          1.650279     9.801322
 4               prcp_annual_mm        224.681732   -16.064132
 5       prcp_growing_season_mm        184.123407    19.938052
 6               prcp_spring_mm         85.879985     3.405301
 7              ohc700_atlantic          1.486911    -7.558566
 8        ohc700_north_atlantic          0.722670    -1.235500
 9       ohc2000_north_atlantic          1.179053     1.129453
 10              ohc700_pacific          1

In [237]:
X_choice_removals_3 = pa_gs[['dtr_annual', 'dtr_spring', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'ohc700_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world',
    'ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic','sst_gulf_stream', 
    'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]
multiple_regression(X_choice_removals_3, y)

['MAE = 8.926758055204589',
 'MSE = 125.47014250310785',
 'RMSE = 11.201345566632067',
 'R Squared = 0.7909419357023244',
 'Adj. R Squared = 0.7875207634052263',
 'Total Data Points: 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618    -7.301648
 1                   dtr_spring          1.463406    -1.928642
 2                  tmax_annual          1.650279    10.385554
 3               prcp_annual_mm        224.681732   -13.268028
 4       prcp_growing_season_mm        184.123407    18.812644
 5              ohc700_atlantic          1.486911    11.418106
 6       ohc2000_north_atlantic          1.179053     7.359042
 7               ohc700_pacific          1.628323     2.016328
 8                 ohc700_world          3.677249   -19.565941
 9              ohc700_natl_djf          0.787782   -30.778206
 10             ohc700_natl_amj          0

In [238]:
y_inland = inland_pa_stations['growing_season_length']
X_inland = inland_pa_stations[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]
y_coastal = coastal_pa_stations['growing_season_length']
X_coastal = coastal_pa_stations[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]
y_lakeside = lakeside_pa_stations['growing_season_length']
X_lakeside = lakeside_pa_stations[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]

In [239]:
multiple_regression(X_inland, y_inland)

['MAE = 7.903038615421362',
 'MSE = 101.94476197632694',
 'RMSE = 10.096769878348567',
 'R Squared = 0.7925760441220471',
 'Adj. R Squared = 0.782068979108142',
 'Total Data Points: 1204',
 '(Reference) predicted variable standard deviation = 22.683614070734397',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.096584    -8.635601
 1                   dtr_spring          1.429845    -0.893934
 2                   dtr_summer          1.273667     1.651865
 3                  tmax_annual          1.493089     4.707797
 4               prcp_annual_mm        218.895351   -17.111908
 5       prcp_growing_season_mm        180.815703    21.427992
 6               prcp_spring_mm         83.766335     3.011035
 7                     latitude          0.585141    -9.348587
 8                    longitude          1.481903     6.981162
 9                  elevation_m        148.665453    -2.810526
 10               dist_coast_km         40.

In [240]:
#removing stations with coefficient < |1|
X_inland_influential = inland_pa_stations[['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'ohc700_atlantic', 'ohc700_south_atlantic', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'cloud_cover_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'dewpoint_2m_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station',
    'cloud_cover_station']]
multiple_regression(X_inland_influential, y_inland)

['MAE = 7.931988565407447',
 'MSE = 102.55423225344262',
 'RMSE = 10.126906351568707',
 'R Squared = 0.7913359731912942',
 'Adj. R Squared = 0.7843446527054355',
 'Total Data Points: 1204',
 '(Reference) predicted variable standard deviation = 22.683614070734397',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.096584    -9.725071
 1                   dtr_summer          1.273667     1.984593
 2                  tmax_annual          1.493089     4.753900
 3               prcp_annual_mm        218.895351   -17.227984
 4       prcp_growing_season_mm        180.815703    21.377788
 5               prcp_spring_mm         83.766335     3.054583
 6                     latitude          0.585141    -8.850426
 7                    longitude          1.481903     8.052666
 8                  elevation_m        148.665453    -2.739983
 9           dist_greatlakes_km         67.291385    -4.471873
 10            dist_atlantic_km        101

In [241]:
#removing stations with coefficient < |1| again
X_inland_influential_2 = inland_pa_stations[['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_djf', 'pna_annual', 'ohc700_atlantic', 'ohc700_south_atlantic', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'cloud_cover_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us','dewpoint_2m_northeast_us',
    'dewpoint_2m_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'cloud_cover_station']]
multiple_regression(X_inland_influential_2, y_inland)

['MAE = 7.944000477310872',
 'MSE = 102.77014145298429',
 'RMSE = 10.137560922282256',
 'R Squared = 0.790896669205379',
 'Adj. R Squared = 0.7846307303545128',
 'Total Data Points: 1204',
 '(Reference) predicted variable standard deviation = 22.683614070734397',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.096584    -9.725071
 1                   dtr_summer          1.273667     1.984593
 2                  tmax_annual          1.493089     4.753900
 3               prcp_annual_mm        218.895351   -17.227984
 4       prcp_growing_season_mm        180.815703    21.377788
 5               prcp_spring_mm         83.766335     3.054583
 6                     latitude          0.585141    -8.850426
 7                    longitude          1.481903     8.052666
 8                  elevation_m        148.665453    -2.739983
 9           dist_greatlakes_km         67.291385    -4.471873
 10            dist_atlantic_km        101.

In [243]:
X_inland_choice_removals = inland_pa_stations[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'ohc700_atlantic', 'ohc700_atlantic_se', 'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 
    'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us','cloud_cover_eastern_us', 'evaporation_eastern_us', 
    'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us','cloud_cover_northeast_us', 'evaporation_northeast_us', 
    'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania','cloud_cover_pennsylvania', 'evaporation_pennsylvania', 
    'dewpoint_station', 'soil_moisture_station','cloud_cover_station', 'evaporation_station']]
multiple_regression(X_inland_choice_removals, y_inland)

['MAE = 7.940446968402912',
 'MSE = 102.66942659559402',
 'RMSE = 10.132592293958838',
 'R Squared = 0.7911015907112081',
 'Adj. R Squared = 0.7831710212472678',
 'Total Data Points: 1204',
 '(Reference) predicted variable standard deviation = 22.683614070734397',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.096584    -8.635601
 1                   dtr_spring          1.429845    -0.893934
 2                   dtr_summer          1.273667     1.651865
 3                  tmax_annual          1.493089     4.707797
 4               prcp_annual_mm        218.895351   -17.111908
 5       prcp_growing_season_mm        180.815703    21.427992
 6               prcp_spring_mm         83.766335     3.011035
 7                     latitude          0.585141    -9.348587
 8                    longitude          1.481903     6.981162
 9                  elevation_m        148.665453    -2.810526
 10               dist_coast_km         40